# Quant Backtester - End-to-End Demo

This notebook demonstrates the capabilities of the **Event-Driven Backtesting Engine**.

## Scenario
We will backtest a naive **Buy & Hold Strategy** on **SPY** (S&P 500 ETF) data.

**Workflow:**
1.  **Data Ingestion**: Download historical data using `yfinance`.
2.  **Initialization**: Setup `DataHandler`, `Portfolio`, `Strategy`, and `BacktestEngine`.
3.  **Execution**: Run the event loop.
4.  **Analysis**: Visualize Equity Curve, Drawdown and calculate Sharpe Ratio.

In [ ]:
# Setup
%load_ext autoreload
%autoreload 2

import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
import os

from src.data_handler import DataHandler
from src.strategy import BuyAndHoldStrategy
from src.portfolio import Portfolio
from src.engine import BacktestEngine
from src.performance import create_equity_curve, calculate_drawdown, calculate_sharpe_ratio

## 1. Data Ingestion
We download the last 2 years of daily data for SPY.

In [ ]:
# Download Data
symbol = "SPY"
csv_path = "data/spy_data.csv"

if not os.path.exists("data"):
    os.makedirs("data")

print(f"Downloading {symbol} data...")
data = yf.download(symbol, start="2022-01-01", end="2023-12-31", progress=False)

# Flatten MultiIndex if present (yfinance update quirk)
if isinstance(data.columns, pd.MultiIndex):
    data.columns = data.columns.get_level_values(0)
    
# Ensure CSV format expects 'Date', 'Close', 'Volume'
data.reset_index(inplace=True)
if 'Date' not in data.columns and 'Datetime' in data.columns:
   data.rename(columns={'Datetime': 'Date'}, inplace=True)

data.to_csv(csv_path, index=False)
print(f"Data saved to {csv_path}")
data.head()

## 2. Initialize Components
We instantiate the core components of our architecture.

In [ ]:
# Components
data_handler = DataHandler(csv_path, symbol)
strategy = BuyAndHoldStrategy()
portfolio = Portfolio(initial_capital=100000.0)
engine = BacktestEngine(data_handler, strategy, portfolio)

## 3. Run Backtest
The engine processes events bar-by-bar.

In [ ]:
engine.run()

## 4. Performance Analysis
We calculate metrics from the portfolio history.

In [ ]:
# Create Curves
equity_curve = create_equity_curve(portfolio.history)
drawdown_curve = calculate_drawdown(equity_curve)
sharpe = calculate_sharpe_ratio(equity_curve)

print(f"Final Equity: ${equity_curve['equity'].iloc[-1]:,.2f}")
print(f"Sharpe Ratio: {sharpe:.2f}")

In [ ]:
# Plotting
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# Equity Curve
ax1.plot(equity_curve.index, equity_curve['equity'], label='Portfolio Equity', color='blue')
ax1.set_title("Equity Curve (Buy & Hold SPY)")
ax1.set_ylabel("Capital ($)")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Drawdown
ax2.fill_between(drawdown_curve.index, drawdown_curve, 0, color='red', alpha=0.3, label='Drawdown')
ax2.set_title("Drawdown")
ax2.set_ylabel("Percentage")
ax2.set_xlabel("Date")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()